<a href="https://colab.research.google.com/github/dialysisdataconsulting-boop/dialysis-data-hipotension/blob/main/Modelo_DIALYSIS_DATA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PREPARAR DATOS

In [2]:
from google.colab import files
upload = files.upload()

Saving test_dialysis.csv to test_dialysis.csv
Saving train_dialysis.csv to train_dialysis.csv


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, recall_score, roc_auc_score
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler

# Cargar datos
train = pd.read_csv('train_dialysis.csv')
test = pd.read_csv('test_dialysis.csv')

# Convertir booleanos
train = train.astype(int)
test = test.astype(int)

# Separar
X_train = train.drop(columns=['Hipotension'])
y_train = train['Hipotension']

X_test = test.drop(columns=['Hipotension'])
y_test = test['Hipotension']

Penalizar FN

In [4]:
# Peso de clases
neg = sum(y_train == 0)
pos = sum(y_train == 1)

scale_pos_weight = neg / pos

Modelo XGBoost clìnico

In [5]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,  # 🔥 CRÍTICO
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

Ajuste de threshold

In [6]:
y_prob = model.predict_proba(X_test)[:,1]

# Threshold bajo (más sensibilidad)
threshold = 0.25

y_pred = (y_prob >= threshold).astype(int)

Evaluaciòn real

In [7]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

recall = recall_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("AUC:", auc)
print("Recall:", recall)
print("FN:", fn)
print("FP:", fp)

AUC: 0.6796818087209907
Recall: 0.8846153846153846
FN: 72
FP: 2379


Cómo interpretar (modo clínica)
🔴 FN → pacientes que NO detectaste (PELIGRO)
🟡 FP → falsas alarmas (aceptables)
🟢 Recall → qué tanto estás detectando

Ajustes

In [8]:
for t in [0.1, 0.2, 0.25, 0.3, 0.4]:
    y_pred = (y_prob >= t).astype(int)
    recall = recall_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    fn = cm[1][0]

    print(f"Threshold {t} → Recall: {recall:.3f} | FN: {fn}")

Threshold 0.1 → Recall: 0.986 | FN: 9
Threshold 0.2 → Recall: 0.923 | FN: 48
Threshold 0.25 → Recall: 0.885 | FN: 72
Threshold 0.3 → Recall: 0.849 | FN: 94
Threshold 0.4 → Recall: 0.732 | FN: 167


Numero de falsos

In [9]:
for t in [0.2]:
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"Threshold {t}")
    print(f"Recall: {recall_score(y_test, y_pred):.3f}")
    print(f"FN: {fn}")
    print(f"FP: {fp}")

Threshold 0.2
Recall: 0.923
FN: 48
FP: 2622


Ajuste DIALYSIS-DATA

Sistema hibrido

Obtener probabilidades del modelo

In [10]:
y_prob = model.predict_proba(X_test)[:, 1]  # probabilidad de hipotensión

Clasificar niveles (semaforo)

In [11]:
import pandas as pd
import numpy as np

df_out = X_test.copy()
df_out["y_true"] = y_test.values
df_out["riesgo"] = y_prob

def clasificar_riesgo(p):
    if p >= 0.4:
        return "ALTO"      # 🔴
    elif p >= 0.2:
        return "MEDIO"     # 🟡
    else:
        return "BAJO"      # 🟢

df_out["nivel"] = df_out["riesgo"].apply(clasificar_riesgo)

Acciones clinicas (sin saturar)

In [12]:
def accion(nivel):
    if nivel == "ALTO":
        return "ALERTA_INMEDIATA"
    elif nivel == "MEDIO":
        return "VIGILANCIA"
    else:
        return "SIN_ACCION"

df_out["accion"] = df_out["nivel"].apply(accion)

FILTRA ANTIFATIGA

sOLO ALERTAS ROJAS

In [13]:
alertas_rojas = df_out[df_out["nivel"] == "ALTO"]

Limitar alertas x turno

In [14]:
TOP_N = 20  # ajusta a tu capacidad operativa
top_alertas = df_out.sort_values("riesgo", ascending=False).head(TOP_N)

ALertar solo cambios importantes

In [16]:
# si tienes histórico por paciente
df_out = df_out.sort_values(["ID_paciente", "Fecha_sesion"])

df_out["riesgo_prev"] = df_out.groupby("ID_paciente")["riesgo"].shift(1)
df_out["delta"] = df_out["riesgo"] - df_out["riesgo_prev"]

alertas_cambio = df_out[(df_out["nivel"] == "ALTO") & (df_out["delta"] > 0.1)]

KeyError: 'ID_paciente'

Resumen operativo por turno

In [17]:
resumen = df_out["nivel"].value_counts()
print(resumen)

nivel
ALTO     2086
MEDIO    1112
BAJO      629
Name: count, dtype: int64


Exportar para Excel/Dashboard

In [18]:
df_out.to_excel("resultado_riesgo_clinico.xlsx", index=False)

SCORE CLÌNICO 0-100

In [19]:
df_out["score"] = (df_out["riesgo"] * 100).round(1)